In [74]:
import numpy as np
from astropy.table import Table
from astropy.table import join
from astropy.table import vstack
from astropy.io import fits

from scipy.spatial import cKDTree
from sklearn.cluster import DBSCAN
import networkx as nx

from scipy.ndimage import gaussian_filter, map_coordinates

import readfof
import readgadget
import MAS_library as MASL
from collections import Counter

from astropy.io import fits
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 360
matplotlib.rcParams['text.usetex'] = True
matplotlib.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}'
# plt.style.use('dark_background')
# import seaborn as sns

import pickle
import gzip

In [2]:
snapdir = '/pscratch/sd/v/vtorresg/quijotes/Halos/FoF/fiducial/0'
snapnum = 3

FoF = readfof.FoF_catalog(snapdir, snapnum, long_ids=False,
                          swap=False, SFR=False, read_IDs=False)
pos = FoF.GroupPos/1e3
pos.shape, type(pos)

((309740, 3), numpy.ndarray)

In [3]:
randv_all = Table.read('/pscratch/sd/v/vtorresg/quijotes/ASTRA/FoF/fiducial/0/group_003_random_voids.fits.gz').to_pandas()
randv_all.shape

(13077247, 5)

In [4]:
randv = randv_all[randv_all['RANDITER']==0]
randv.shape

(130670, 5)

In [5]:
pos_rand = randv[['X', 'Y', 'Z']].to_numpy()
pos_rand.shape

(130670, 3)

In [6]:
prob = Table.read('/pscratch/sd/v/vtorresg/quijotes/ASTRA/FoF/fiducial/0/group_003_probability.fits.gz').to_pandas()
prob.shape

(309740, 4)

In [7]:
assert len(pos) == len(prob)

In [8]:
Lbox = 1000.0

Nh = len(pos)
Nr = len(pos_rand)

### 1. Halo-Halo

In [9]:
k_hh = 8

tree_h = cKDTree(pos, boxsize=Lbox)
dist_hh, idx_hh = tree_h.query(pos, k=k_hh + 1)

# quitar self
dist_hh = dist_hh[:, 1:]
idx_hh  = idx_hh[:, 1:]

src_hh = np.repeat(np.arange(Nh), k_hh)
dst_hh = idx_hh.ravel()
d_hh = dist_hh.ravel()

### 2. Random-Random

In [10]:
k_rr = 8

tree_r = cKDTree(pos_rand, boxsize=Lbox)
dist_rr, idx_rr = tree_r.query(pos_rand, k=k_rr + 1)

dist_rr = dist_rr[:, 1:]
idx_rr = idx_rr[:, 1:]

src_rr = np.repeat(np.arange(Nr), k_rr)
dst_rr = idx_rr.ravel()
d_rr = dist_rr.ravel()

## Grafo

In [11]:
G = nx.Graph()

In [12]:
prob_arr = prob[['PVOID', 'PSHEET', 'PFILAMENT', 'PKNOT']].to_numpy()

In [13]:
Nh = len(pos)
Nr = len(pos_rand)
assert len(prob_arr) == Nh

In [14]:
for i in range(Nh):
    G.add_node(i, node_type='halo', pos=pos[i],
               PVOID=float(prob_arr[i, 0]),
               PSHEET=float(prob_arr[i, 1]),
               PFILAMENT=float(prob_arr[i, 2]),
               PKNOT=float(prob_arr[i, 3]))

In [15]:
for i in range(Nr):
    node_id = Nh + i
    G.add_node(node_id, node_type='random', pos=pos_rand[i],
               PVOID=1.0, PSHEET=0.0, PFILAMENT=0.0, PKNOT=0.0)

Creo que la idea seria algo como:

-> Halos = informacion

-> random voids = puntos donde reconstruir el campo DM

In [16]:
print('Nh =', Nh)
print('Nr =', Nr)

Nh = 309740
Nr = 130670


In [17]:
Lbox = 1000.0

### edges halos

In [18]:
k_hh = 8

In [19]:
tree_h = cKDTree(pos, boxsize=Lbox)

In [20]:
dist_hh, idx_hh = tree_h.query(pos, k=k_hh + 1)

# quitar self
dist_hh = dist_hh[:, 1:]
idx_hh = idx_hh[:, 1:]

src_hh = np.repeat(np.arange(Nh), k_hh)
dst_hh = idx_hh.ravel()
d_hh = dist_hh.ravel()

In [21]:
G.add_edges_from((int(i), int(j),
                  {'edge_type': 'HH', 'distance': float(d)})
                 for i, j, d in zip(src_hh, dst_hh, d_hh))

### edges rand

In [22]:
k_rr = 8

In [23]:
tree_r = cKDTree(pos_rand, boxsize=Lbox)

In [24]:
dist_rr, idx_rr = tree_r.query(pos_rand, k=k_rr + 1)

dist_rr = dist_rr[:, 1:]
idx_rr = idx_rr[:, 1:]

src_rr = np.repeat(np.arange(Nr), k_rr)
dst_rr = idx_rr.ravel()
d_rr = dist_rr.ravel()

In [25]:
G.add_edges_from((Nh + int(i), Nh + int(j),
                  {'edge_type': 'RR', 'distance': float(d)})
                 for i, j, d in zip(src_rr, dst_rr, d_rr))

----------------

In [26]:
print('Nodes:', G.number_of_nodes())
print('Edges:', G.number_of_edges())

Nodes: 440410
Edges: 2162502


In [27]:
print(Counter(data['node_type'] for _, data in G.nodes(data=True)))
print(Counter(data['edge_type'] for _, _, data in G.edges(data=True)))

Counter({'halo': 309740, 'random': 130670})
Counter({'HH': 1532295, 'RR': 630207})


halos                         random voids

●──●──●                       ○──○──○
│╲ │                          │ ╲  │
●─●──●         +              ○──○─○
   ╲                             ╲
    ●                             ○

In [28]:
G.nodes[0]

{'node_type': 'halo',
 'pos': array([686.011  , 533.36017, 426.34787], dtype=float32),
 'PVOID': 0.0,
 'PSHEET': 0.14000000059604645,
 'PFILAMENT': 0.6599999666213989,
 'PKNOT': 0.19999998807907104}

In [30]:
G.nodes[Nh+1]

{'node_type': 'random',
 'pos': array([668.1629 , 342.78842, 301.57013], dtype=float32),
 'PVOID': 1.0,
 'PSHEET': 0.0,
 'PFILAMENT': 0.0,
 'PKNOT': 0.0}

In [89]:
with gzip.open('graph_fid_0.pkl.gz', 'wb') as f:
    pickle.dump(G, f, protocol=pickle.HIGHEST_PROTOCOL)

In [90]:
with gzip.open('graph_fid_0.pkl.gz', 'rb') as f:
    G = pickle.load(f)

In [44]:
G

In [41]:
print(G.number_of_nodes())
print(G.number_of_edges())

print(G.nodes[0])
print(G.edges[list(G.edges())[0]])

440410
2162502
{'node_type': 'halo', 'pos': array([686.011  , 533.36017, 426.34787], dtype=float32), 'PVOID': 0.0, 'PSHEET': 0.14000000059604645, 'PFILAMENT': 0.6599999666213989, 'PKNOT': 0.19999998807907104}
{'edge_type': 'HH', 'distance': 4.349417098210152}


----

## DM

In [54]:
ptype = [1] #dm particles
snapshot = '/pscratch/sd/v/vtorresg/quijotes/Snapshots/fiducial/0/snapdir_003/snap_003'

In [63]:
pos = readgadget.read_block(snapshot, 'POS ', ptype)/1e3 # originalm estan en Kpc
pos.shape

(134217728, 3)

In [64]:
BoxSize = 1000.0   # Mpc/h
Ngrid = 512

In [65]:
density = np.zeros((Ngrid, Ngrid, Ngrid), dtype=np.float32)
density.shape

(512, 512, 512)

In [66]:
MASL.MA(pos.astype(np.float32), density, BoxSize, MAS='CIC', verbose=True)


Using CIC mass assignment scheme
Time taken = 2.976 seconds



$$ \delta_{\rm DM}(\mathbf{x}) = \frac{\rho_{\rm DM}(\mathbf{x})}{\bar{\rho}_{\rm DM}}-1$$

In [67]:
delta = density / np.mean(density) - 1.0 # density contrast
delta.shape

(512, 512, 512)

In [69]:
dx = BoxSize / Ngrid

In [70]:
R = 10.0  # Mpc/h
sigma_grid = R / dx

In [73]:
delta_10 = gaussian_filter(delta, sigma=sigma_grid, mode='wrap') # wrap porque es periodica
delta_10.min(), delta_10.max(), delta_10.mean()

(-0.6994475, 3.833689, -2.6223943e-06)

necesito mapear random_voids a dm

In [75]:
def sample_periodic_field(field, positions, BoxSize):
    Ngrid = field.shape[0]
    dx = BoxSize / Ngrid
    coords = (positions / dx).T # coord fsicas -> coord grid
    values = map_coordinates(field, coords, order=1, mode='wrap')
    return values

In [77]:
delta_dm_rand = sample_periodic_field(delta_10, pos_rand, BoxSize=BoxSize)
delta_dm_rand.shape

(130670,)

$$ \mathbf{x}_i=(x_i,y_i,z_i) \quad\rightarrow\quad \delta_{{\rm DM},i}$$

In [78]:
for i in range(10):
    print(pos_rand[i], delta_dm_rand[i])

[193.29807  95.61732 731.27606] -0.26673192
[668.1629  342.78842 301.57013] -0.38530618
[519.1762 800.7633 640.4621] -0.0781904
[186.4669  641.81604 328.5778 ] -0.27330074
[588.6219  126.72455 466.7231 ] -0.06958866
[390.08142 830.25024 144.86679] -0.121532306
[628.40796 552.6343  859.5213 ] -0.30187863
[857.081   705.81964 964.7118 ] 0.011649962
[864.8408  512.03595  48.60765] -0.23782718
[891.704    116.886185 974.98456 ] -0.24892132


In [86]:
with open('delta_dm_randoms_R10.pkl', 'wb') as f:
    pickle.dump(delta_dm_rand, f)

In [87]:
with open('delta_dm_randoms_R10.pkl', 'rb') as f:
    delta_dm_rand = pickle.load(f)

In [88]:
delta_dm_rand.shape

(130670,)

inicialmente la densidad relativa, pero tal vez pueda meter mas info usando varias escalas

- $\delta_5$: rho local, sensible a estructura pequeña
- $\delta_{10}$: entorno medio
- $\delta_{20}$: underdensity gran escala

In [83]:
scales = [5., 10., 20.]
targets = []

In [84]:
for R in scales:
    sigma_grid = R / dx
    delta_R = gaussian_filter(delta, sigma=sigma_grid, mode='wrap')
    y = sample_periodic_field(delta_R, pos_rand, BoxSize)
    targets.append(y)

In [85]:
targets = np.column_stack(targets)
targets.shape

(130670, 3)